In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
from datetime import date, timedelta

# ---------- CONFIG ----------
BASE_URL = "https://www.sharesansar.com/category/latest"
TOKEN = "kXv0PoCD9Ioz0uZo3cjdZ5Qm6A7eoPD5168Ag9Wi"  # Replace with valid token
OUTPUT_CSV = "sharesansar_news.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}

# ---------- GENERATE ALL DATES IN 2024 ----------
start_date = date(2012, 1, 1)
end_date = date(2025, 12, 25)
delta = timedelta(days=1)

all_dates_2024 = []
current = start_date
while current <= end_date:
    all_dates_2024.append(current.strftime("%Y-%m-%d"))
    current += delta

# ---------- SCRAPER ----------
all_data = []

for date_str in all_dates_2024:
    next_url = f"{BASE_URL}?company=&date={date_str}&_token={TOKEN}"
    print(f"\nScraping news for {date_str}")

    while next_url:
        response = requests.get(next_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")

        # ---------- SCRAPE NEWS ----------
        news_blocks = soup.select("div.featured-news-list")
        if not news_blocks:
            if next_url.endswith(f"date={date_str}&_token={TOKEN}"):
                print(f"No news found for {date_str}")
            break

        for block in news_blocks:
            try:
                a_tag = block.select_one("div.col-md-10 a")
                headline = a_tag.select_one("h4.featured-news-title").get_text(strip=True)
                link = a_tag["href"]
                all_data.append((date_str, headline, link))
            except:
                continue

        # ---------- HANDLE PAGINATION ----------
        next_tag = soup.select_one("ul.pagination li a[rel='next']")
        if next_tag:
            next_url = urljoin(BASE_URL, next_tag["href"])
        else:
            next_url = None  # No more pages

# ---------- SAVE TO CSV ----------
if all_data:
    df = pd.DataFrame(all_data, columns=["Date", "Headline", "Link"])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved {len(all_data)} headlines to {OUTPUT_CSV}")
else:
    print("⚠️ No data scraped.")



Scraping news for 2012-01-01

Scraping news for 2012-01-02

Scraping news for 2012-01-03

Scraping news for 2012-01-04

Scraping news for 2012-01-05

Scraping news for 2012-01-06

Scraping news for 2012-01-07

Scraping news for 2012-01-08

Scraping news for 2012-01-09

Scraping news for 2012-01-10

Scraping news for 2012-01-11

Scraping news for 2012-01-12

Scraping news for 2012-01-13

Scraping news for 2012-01-14

Scraping news for 2012-01-15

Scraping news for 2012-01-16

Scraping news for 2012-01-17

Scraping news for 2012-01-18

Scraping news for 2012-01-19

Scraping news for 2012-01-20

Scraping news for 2012-01-21

Scraping news for 2012-01-22

Scraping news for 2012-01-23

Scraping news for 2012-01-24

Scraping news for 2012-01-25

Scraping news for 2012-01-26

Scraping news for 2012-01-27

Scraping news for 2012-01-28

Scraping news for 2012-01-29

Scraping news for 2012-01-30

Scraping news for 2012-01-31

Scraping news for 2012-02-01

Scraping news for 2012-02-02

Scraping 

ANALYSZE sentiment

In [2]:
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Download VADER lexicon if not already done
nltk.download('vader_lexicon')

# Load your CSV
df = pd.read_csv("sharesansar_news.csv")

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Apply sentiment analysis to each headline
df['sentiment'] = df['Headline'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

# Function to classify sentiment
def classify_sentiment(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"

# Calculate average sentiment per day
daily_sentiment = df.groupby('Date')['sentiment'].mean().reset_index()

# Classify daily sentiment
daily_sentiment['sentiment_label'] = daily_sentiment['sentiment'].apply(classify_sentiment)

# Display the daily sentiment result
print(daily_sentiment)
df.to_csv('Sentiment-collected.csv', index=False)


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\rojin\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


            Date  sentiment sentiment_label
0     2012-01-01  -0.123233        Negative
1     2012-01-02   0.001760         Neutral
2     2012-01-03  -0.048720         Neutral
3     2012-01-04   0.136890        Positive
4     2012-01-05   0.177810        Positive
...          ...        ...             ...
4429  2025-12-21   0.088611        Positive
4430  2025-12-22   0.233822        Positive
4431  2025-12-23   0.019172         Neutral
4432  2025-12-24   0.100626        Positive
4433  2025-12-25   0.087800        Positive

[4434 rows x 3 columns]


In [15]:
# import pandas as pd

# # Load both datasets
# stock_df = pd.read_csv("NEPSE_Ta_data.csv")
# sentiment_df = pd.read_csv("Sentiment-collected.csv")

# # Convert date columns to datetime
# stock_df['date'] = pd.to_datetime(stock_df['date'])
# sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])

# # Step 1: aggregate sentiment per day
# daily_sentiment = sentiment_df.groupby('Date', as_index=False)['sentiment'].mean()

# # Step 2: reindex sentiment to cover all calendar days
# # This ensures no missing days (for forward-fill to work correctly)
# date_range = pd.date_range(daily_sentiment['Date'].min(), stock_df['date'].max())
# daily_sentiment = (
#     daily_sentiment
#     .set_index('Date')
#     .reindex(date_range)
#     .fillna(method='ffill')  # fill missing dates with previous day's sentiment
#     .rename_axis('Date')
#     .reset_index()
# )

# # Step 3: merge so that each trading day gets sentiment up to that date
# merged_df = pd.merge_asof(
#     stock_df.sort_values('date'),
#     daily_sentiment.sort_values('Date'),
#     left_on='date',
#     right_on='Date',
#     direction='backward'
# )

# # Drop duplicate 'Date' column if you want
# merged_df.drop(columns=['Date'], inplace=True)

# Save the merged dataset
merged_df.to_csv('stock_data_with_sentiment.csv', index=False)
print("✅ Saved as stock_with_sentiment.csv")


✅ Saved as stock_with_sentiment.csv


Some redundancy checking 

In [30]:
sentiment_df.groupby('Date').sentiment.mean().head(10)

Date
2012-01-01   -0.123233
2012-01-02    0.001760
2012-01-03   -0.048720
2012-01-04    0.136890
2012-01-05    0.177810
2012-01-06    0.026490
2012-01-07    0.030586
2012-01-08    0.013783
2012-01-09   -0.042970
2012-01-10   -0.049040
Name: sentiment, dtype: float64

In [3]:
import pandas as pd
import numpy as np

# Load both datasets
stock_df = pd.read_csv("Nepse_Ta_data.csv")
sentiment_df = pd.read_csv("Sentiment-Collected.csv")

# Convert date columns to datetime
stock_df['date'] = pd.to_datetime(stock_df['date'])
sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])

# Step 1: Aggregate sentiment per calendar day
daily_sentiment = sentiment_df.groupby('Date', as_index=False)['sentiment'].mean()

# Step 2: Create complete date range for sentiment (all calendar days)
date_range = pd.date_range(
    min(daily_sentiment['Date'].min(), stock_df['date'].min()),
    max(daily_sentiment['Date'].max(), stock_df['date'].max())
)

daily_sentiment_complete = (
    daily_sentiment
    .set_index('Date')
    .reindex(date_range)
    .fillna(method='ffill')
    .rename_axis('Date')
    .reset_index()
)

# Step 3: For each trading day, get ALL sentiment until next trading day
def get_accumulated_sentiment(trading_date, daily_sentiment_complete, stock_df):
    """
    For a trading date, get aggregated sentiment from THIS trading day 
    until the day before next trading day
    """
    # Find the next trading day
    all_trading_dates = sorted(stock_df['date'].unique())
    current_idx = all_trading_dates.index(trading_date)
    
    if current_idx == len(all_trading_dates) - 1:
        # Last trading day, just use today's sentiment
        mask = (daily_sentiment_complete['Date'] == trading_date)
        day_sentiment = daily_sentiment_complete.loc[mask, 'sentiment'].values
        return day_sentiment[0] if len(day_sentiment) > 0 else np.nan
    
    next_trading_day = all_trading_dates[current_idx + 1]
    
    # Get ALL sentiments from this trading day up to (but not including) next trading day
    start_date = trading_date
    end_date = next_trading_day - pd.Timedelta(days=1)
    
    mask = (daily_sentiment_complete['Date'] >= start_date) & (daily_sentiment_complete['Date'] <= end_date)
    accumulated_sentiments = daily_sentiment_complete.loc[mask, 'sentiment'].dropna()
    
    if len(accumulated_sentiments) > 0:
        return np.mean(accumulated_sentiments)
    
    return np.nan

# Step 4: Apply to each trading day
print("Calculating accumulated sentiment...")
stock_df['accumulated_sentiment'] = stock_df['date'].apply(
    lambda x: get_accumulated_sentiment(x, daily_sentiment_complete, stock_df)
)

# Step 5: Forward fill any missing values
stock_df['accumulated_sentiment'] = stock_df['accumulated_sentiment'].fillna(method='ffill')

# Save the result
stock_df.to_csv('stock_with_sentiment-aggregated.csv', index=False)
print("✅ Saved as stock_with_sentiment.csv")

# Show detailed example
print("\n=== DETAILED EXAMPLE ===")
sample_trading_dates = ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04', '2024-01-08']
sample_df = stock_df[stock_df['date'].isin(pd.to_datetime(sample_trading_dates))]

for _, row in sample_df.iterrows():
    # Show what dates contributed to this sentiment
    all_trading_dates = sorted(stock_df['date'].unique())
    current_idx = all_trading_dates.index(row['date'])
    
    if current_idx < len(all_trading_dates) - 1:
        next_trading_day = all_trading_dates[current_idx + 1]
        start_date = row['date']
        end_date = next_trading_day - pd.Timedelta(days=1)
        
        # Get the actual dates that were aggregated
        mask = (daily_sentiment_complete['Date'] >= start_date) & (daily_sentiment_complete['Date'] <= end_date)
        contributing_dates = daily_sentiment_complete.loc[mask, ['Date', 'sentiment']]
        
        print(f"📅 {row['date'].strftime('%Y-%m-%d (%a)')}:")
        print(f"   📊 Accumulated sentiment: {row['accumulated_sentiment']:.4f}")
        print(f"   🔗 Used to predict: {next_trading_day.strftime('%Y-%m-%d (%a)')}")
        print(f"   📰 Contributing news days ({len(contributing_dates)}):")
        for _, contrib in contributing_dates.iterrows():
            print(f"      • {contrib['Date'].strftime('%Y-%m-%d (%a)')}: sentiment = {contrib['sentiment']:.4f}")
        print()

C:\Users\rojin\AppData\Local\Temp\ipykernel_19360\3157334348.py:25: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='ffill')


Calculating accumulated sentiment...
✅ Saved as stock_with_sentiment.csv

=== DETAILED EXAMPLE ===
📅 2024-01-01 (Mon):
   📊 Accumulated sentiment: 0.2798
   🔗 Used to predict: 2024-01-02 (Tue)
   📰 Contributing news days (1):
      • 2024-01-01 (Mon): sentiment = 0.2798

📅 2024-01-02 (Tue):
   📊 Accumulated sentiment: 0.2213
   🔗 Used to predict: 2024-01-03 (Wed)
   📰 Contributing news days (1):
      • 2024-01-02 (Tue): sentiment = 0.2213

📅 2024-01-03 (Wed):
   📊 Accumulated sentiment: 0.2949
   🔗 Used to predict: 2024-01-04 (Thu)
   📰 Contributing news days (1):
      • 2024-01-03 (Wed): sentiment = 0.2949

📅 2024-01-04 (Thu):
   📊 Accumulated sentiment: 0.2829
   🔗 Used to predict: 2024-01-07 (Sun)
   📰 Contributing news days (3):
      • 2024-01-04 (Thu): sentiment = 0.1730
      • 2024-01-05 (Fri): sentiment = 0.3379
      • 2024-01-06 (Sat): sentiment = 0.3379

📅 2024-01-08 (Mon):
   📊 Accumulated sentiment: 0.1634
   🔗 Used to predict: 2024-01-09 (Tue)
   📰 Contributing news da

C:\Users\rojin\AppData\Local\Temp\ipykernel_19360\3157334348.py:67: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  stock_df['accumulated_sentiment'] = stock_df['accumulated_sentiment'].fillna(method='ffill')


In [16]:
import pandas as pd
df=pd.read_csv('stock_with_sentiment-aggregated.csv')
df.describe()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df = df[(df['date'] > '2012-01-01') & (df['date'] < '2022-12-31')].reset_index(drop=True)


In [17]:
df.describe()

,o,h,l,c,v,date,psar,PosDI,NegDI,ADX,accumulated_sentiment
count,2516.000000,2516.000000,2516.000000,2516.000000,2.516000e+03,2516,2516.000000,2516.000000,2516.000000,2516.000000,2516.000000
mean,1354.164654,1362.126618,1344.540274,1352.786948,1.357641e+09,2017-07-04 19:20:07.631160576,1349.565882,40.828901,33.077405,44.795808,0.196337
min,299.000000,299.000000,299.000000,299.000000,0.000000e+00,2012-01-02 00:00:00,299.000000,0.387490,0.154763,9.371174,-0.243070
25%,918.000000,918.000000,918.000000,918.000000,0.000000e+00,2014-09-22 18:00:00,903.957198,20.398411,17.420912,31.163141,0.106563
50%,1250.410000,1257.430000,1242.935000,1250.905000,3.111333e+08,2017-07-07 12:00:00,1244.167431,37.477644,31.138839,42.800835,0.200309
75%,1654.867500,1666.960000,1647.750000,1653.267500,1.193020e+09,2020-02-18 06:00:00,1650.329485,58.470752,45.048221,56.431180,0.286416
max,3208.530000,3227.110000,3178.250000,3198.190000,2.164760e+10,2022-12-29 00:00:00,3227.110000,99.152349,97.573701,95.271091,0.656932
std,677.334180,684.459817,667.148095,675.052300,2.742433e+09,NaN,677.659778,24.853357,20.146822,17.198408,0.128022
